# 03 — Pilot experiment (pipeline validation)

This notebook runs the **offline CPU smoke test**: a tiny synthetic dataset
and a tiny randomly initialized model. It validates the pipeline end-to-end
(config → data → alignment → training → evaluation → artifacts).

**This is not a research result.** Metrics shown here come from a random
model. Real training runs happen on Colab/Kaggle (see
`docs/experiment_protocol.md`).

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))
print("repo root:", repo_root)

In [ ]:
from axner.config.loader import load_experiment_config
from axner.config.validation import validate_or_raise

cfg = validate_or_raise(load_experiment_config(
    "smoke_test", config_dir=repo_root / "configs" / "experiments"
))
cfg["experiment"]["id"], cfg["data"]["dataset"]["name"], cfg["model"]["tiny_random"]

## Dry run (data pipeline only, no model)

Exercises data preparation, IOB2 audits, duplicates detection, statistics,
and artifact writing — useful as a preflight before any cloud run.

In [ ]:
from axner.experiments.runner import execute_run

summary_dry = execute_run(cfg, run_label="notebook-dry", dry_run=True)
summary_dry

## Full smoke run (tiny random model, CPU, offline)

In [ ]:
summary = execute_run(cfg, run_label="notebook-smoke")
summary

In [ ]:
from pathlib import Path

from axner.artifacts.manager import load_json, load_manifest

run_dir = Path(summary["run_dir"])
print(sorted(p.name for p in run_dir.rglob("*") if p.is_file()))
load_manifest(run_dir)

In [ ]:
load_json(run_dir / "metrics" / "test_metrics_jv.json")

In [ ]:
# Sentence-level predictions (word-level gold vs predicted tags for kept words).
from axner.artifacts.manager import load_jsonl

load_jsonl(run_dir / "predictions" / "test_jv_sentence_level.jsonl")[0]

**Next steps:** real experiments run on Colab/Kaggle GPUs:
`docs/experiment_protocol.md` documents the run matrix, and
`docs/reproducibility.md` explains how bundles move back to this repository
for analysis in notebook 04.